# Taylor Root Prediction — Reviewer Demo Notebook

이 노트북은 **git clone 이후, 상대경로만으로** 아래를 한 번에 재현할 수 있도록 구성했습니다.

1. (선택) 필요한 패키지 설치  
2. 데이터셋 생성 (Root regressor용 / Interval(Transformer)용)  
3. (선택) 모델 학습 (ANN / LSTM / MLP / Transformer interval)  
4. 평가 실행 (baseline 포함, 실패 func_id 분포/그림 생성)

> ⚠️ 기본값은 **빠른 재현(샘플 2만)** 기준입니다.  
> 시간이 충분한 환경이라면 `N_TOTAL_*`를 늘리면 됩니다.


In [ ]:
# (선택) 최소 의존성 설치
# - 이미 환경이 준비되어 있으면 이 셀은 건너뛰어도 됩니다.
#
# !pip install -U pyyaml numpy torch tqdm matplotlib sympy requests
#
# 권장:
# - torch는 CUDA/CPU 환경에 맞는 wheel 설치가 필요할 수 있습니다.


## 0) 공통 유틸 / 레포 루트 자동 인식

In [ ]:
from __future__ import annotations

from pathlib import Path
import os, sys, subprocess, shutil
import re
import json
import numpy as np

def find_repo_root(start: Path | None = None) -> Path:
    """현재 노트북 위치가 어디든, configs/ 폴더를 기준으로 레포 루트를 찾습니다."""
    if start is None:
        start = Path.cwd().resolve()
    else:
        start = start.resolve()

    for p in [start] + list(start.parents):
        if (p / "configs").is_dir():
            return p
    raise RuntimeError("Could not find repo root (missing configs/). Run this notebook from inside the repo.")

REPO = find_repo_root()
print("REPO_ROOT =", REPO)

def R(rel: str | os.PathLike | None) -> Path | None:
    """레포 기준 상대경로를 절대경로로 변환"""
    if rel is None:
        return None
    s = str(rel).strip()
    if not s:
        return None
    s = os.path.expanduser(os.path.expandvars(s))
    p = Path(s)
    if p.is_absolute():
        return p
    return (REPO / p).resolve()

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def run(cmd, env=None, cwd=None):
    """subprocess 실행 + stdout/stderr 표시"""
    print(">>", " ".join(map(str, cmd)))
    r = subprocess.run(cmd, env=env, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if r.stdout:
        print("----- STDOUT -----")
        print(r.stdout)
    if r.stderr:
        print("----- STDERR -----")
        print(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed (code={r.returncode})")
    return r

def find_generator(keywords: list[str], search_dirs: list[str] | None = None) -> Path | None:
    """레포 안에서 '데이터 생성기 스크립트'를 키워드로 탐색"""
    if search_dirs is None:
        search_dirs = ["data_generation", "scripts", "tools", "datasets", "dataset", "data", "models", "."]
    roots = [R(d) for d in search_dirs]
    roots = [p for p in roots if p is not None and p.exists()]

    kws = [k.lower() for k in keywords]
    for root in roots:
        for py in root.rglob("*.py"):
            try:
                txt = py.read_text(encoding="utf-8", errors="ignore")
            except Exception:
                continue
            low = txt.lower()
            if all(k in low for k in kws):
                return py
    return None

def migrate_dataset_dir(expected_dir: Path, candidates: list[Path], marker_files: list[str]) -> bool:
    """잘못 생성된 데이터 폴더를 data/... 표준 위치로 이동"""
    ensure_dir(expected_dir.parent)
    expected_has = all((expected_dir / f).exists() for f in marker_files)

    if expected_has:
        return False

    for cand in candidates:
        if cand.exists() and all((cand / f).exists() for f in marker_files):
            print(f"[MIGRATE] Found dataset at {cand.relative_to(REPO) if cand.is_relative_to(REPO) else cand}")
            print(f"          -> moving to {expected_dir.relative_to(REPO)}")
            if expected_dir.exists():
                # merge move (if empty) else raise
                if any(expected_dir.iterdir()):
                    raise RuntimeError(f"Expected dir already exists and not empty: {expected_dir}")
                expected_dir.rmdir()
            shutil.move(str(cand), str(expected_dir))
            return True
    return False


## 1) 경로 설정 + (필요시) 잘못 생성된 데이터 폴더 자동 이동

In [ ]:
# ---- 리뷰어용 기본값(필요하면 수정) ----
DEGREE = 25

# 빠른 재현용(작게): 20000~100000 정도 추천(환경에 따라 조절)
N_TOTAL_ROOT = 20000
N_TOTAL_INTERVAL = 20000

# 표준 데이터 저장 위치(상대경로)
OUT_ROOT_DIR = R("data/taylor_data_physchem_v4_deg25")
OUT_INTERVAL_DIR = R("data/taylor_data_physchem_v4_interval")

ensure_dir(OUT_ROOT_DIR)
ensure_dir(OUT_INTERVAL_DIR)

# Root-regression NPZ (ann/lstm/mlp)
train_npz = OUT_ROOT_DIR / f"taylor_deg{DEGREE}_train.npz"
val_npz   = OUT_ROOT_DIR / f"taylor_deg{DEGREE}_val.npz"
test_npz  = OUT_ROOT_DIR / f"taylor_deg{DEGREE}_test.npz"

# Interval(Transformer) NPZ
interval_train_npz = OUT_INTERVAL_DIR / f"taylor_deg{DEGREE}_train.npz"
interval_val_npz   = OUT_INTERVAL_DIR / f"taylor_deg{DEGREE}_val.npz"
interval_test_npz  = OUT_INTERVAL_DIR / f"taylor_deg{DEGREE}_test.npz"

# ---- "경로 불일치" 자동 정리 ----
# 과거 노트북/실행에서 data/ 없이 레포 루트에 생성되거나, root/ 아래로 생성된 경우를 흡수합니다.
migrate_dataset_dir(
    expected_dir=OUT_ROOT_DIR,
    candidates=[R("taylor_data_physchem_v4_deg25"), R("root/taylor_data_physchem_v4_deg25")],
    marker_files=[f"taylor_deg{DEGREE}_train.npz", f"taylor_deg{DEGREE}_val.npz", f"taylor_deg{DEGREE}_test.npz"],
)

migrate_dataset_dir(
    expected_dir=OUT_INTERVAL_DIR,
    candidates=[R("taylor_data_physchem_v4_interval"), R("root/taylor_data_physchem_v4_interval")],
    marker_files=[f"taylor_deg{DEGREE}_train.npz", f"taylor_deg{DEGREE}_val.npz", f"taylor_deg{DEGREE}_test.npz"],
)

print("[ROOT NPZ paths]")
print(" train:", train_npz.relative_to(REPO))
print(" val  :", val_npz.relative_to(REPO))
print(" test :", test_npz.relative_to(REPO))

print("\n[INTERVAL NPZ paths]")
print(" train:", interval_train_npz.relative_to(REPO))
print(" val  :", interval_val_npz.relative_to(REPO))
print(" test :", interval_test_npz.relative_to(REPO))


## 2) 데이터 생성기(스크립트) 자동 탐색

In [ ]:
# root generator: coeffs/root0/root1/root2/template_str/expr_str/roots 등을 포함한 생성기
gen_root = find_generator(["generate_dataset", "templates.json"])

# interval generator: expr_str & roots 저장 형태(또는 interval data) 키워드로 탐색
# 프로젝트에 따라 키워드가 다를 수 있어, 필요하면 아래 키워드를 수정하세요.
gen_interval = find_generator(["expr_str", "roots", "taylor_deg25"])

print("[FOUND GENERATORS]")
print(" root_gen     =", (gen_root.relative_to(REPO) if gen_root else None))
print(" interval_gen =", (gen_interval.relative_to(REPO) if gen_interval else None))

if gen_root is None:
    raise RuntimeError(
        "Root dataset generator script not found automatically.\n"
        "→ 생성기 .py 파일을 repo 내부(data_generation/ 등)에 두고 다시 실행하세요."
    )

if gen_interval is None:
    raise RuntimeError(
        "Interval dataset generator script not found automatically.\n"
        "→ interval 생성기 .py 파일을 repo 내부(data_generation/ 등)에 두고 다시 실행하세요."
    )


## 3) Root regression 데이터셋 생성 (ann/lstm/mlp)

In [ ]:
need_root = (not train_npz.exists()) or (not val_npz.exists()) or (not test_npz.exists())

if not need_root:
    print("[SKIP] root npz already exists.")
else:
    cmd = [
        sys.executable, str(gen_root),
        "--degree", str(DEGREE),
        "--n-total", str(N_TOTAL_ROOT),
        "--seed", "42",
        "--out-dir", os.path.relpath(str(OUT_ROOT_DIR), str(REPO)),  # 항상 상대경로
        "--save-expr-str", "1",
    ]
    run(cmd, env={"PYTHONPATH": str(REPO), **os.environ}, cwd=str(REPO))

print("[ROOT DATA CHECK]")
for p in [train_npz, val_npz, test_npz]:
    print(" ", p.relative_to(REPO), "exists=", p.exists())


## 4) Interval(Transformer) 데이터셋 생성

In [ ]:
need_interval = (not interval_train_npz.exists()) or (not interval_val_npz.exists()) or (not interval_test_npz.exists())

if not need_interval:
    print("[SKIP] interval npz already exists.")
else:
    cmd = [
        sys.executable, str(gen_interval),
        "--degree", str(DEGREE),
        "--n-total", str(N_TOTAL_INTERVAL),
        "--seed", "42",
        "--out-dir", os.path.relpath(str(OUT_INTERVAL_DIR), str(REPO)),  # 항상 상대경로
        "--save-expr-str", "1",
    ]
    run(cmd, env={"PYTHONPATH": str(REPO), **os.environ}, cwd=str(REPO))

print("[INTERVAL DATA CHECK]")
for p in [interval_train_npz, interval_val_npz, interval_test_npz]:
    print(" ", p.relative_to(REPO), "exists=", p.exists())


## 5) (선택) 데이터 키/형태 빠른 점검

In [ ]:
import numpy as np

def peek_npz(npz_path: Path, keys_top: int = 20):
    z = np.load(npz_path, allow_pickle=True)
    keys = list(z.keys())
    print(f"[NPZ] {npz_path.relative_to(REPO)}")
    print(" keys:", keys[:keys_top], ("..." if len(keys)>keys_top else ""))
    for k in keys[:min(8, len(keys))]:
        v = z[k]
        try:
            shape = v.shape
        except Exception:
            shape = "?"
        print(f"  - {k:12s} dtype={getattr(v,'dtype',None)} shape={shape}")
    print()

peek_npz(train_npz)
peek_npz(interval_train_npz)


## 6) (선택) 모델 학습 실행
- 기본은 **주석 처리**입니다.
- 실행하면 `results/` 아래에 출력이 생깁니다.
- 모든 경로는 **상대경로**만 사용합니다.

In [ ]:
import torch

def train_script(script_rel: str, cfg_rel: str, out_rel: str,
                 train_path: Path, val_path: Path, test_path: Path | None,
                 extra_env: dict | None = None):
    script = R(script_rel)
    assert script and script.exists(), f"Script not found: {script_rel}"
    env = os.environ.copy()
    env.update({
        "TAYLOR_CFG": str(R(cfg_rel)),                 # 상대경로 -> 절대
        "TRAIN_NPZ": str(train_path),
        "VAL_NPZ": str(val_path),
        "TEST_NPZ": (str(test_path) if test_path is not None else ""),
        "OUT_DIR": str(R(out_rel)),
        "DEVICE": ("cuda" if torch.cuda.is_available() else "cpu"),
        "PYTHONPATH": str(REPO),
    })
    if extra_env:
        env.update({k: str(v) for k,v in extra_env.items()})
    ensure_dir(R(out_rel))
    run([sys.executable, str(script)], env=env, cwd=str(REPO))

# ---- 필요한 것만 주석 해제해서 실행하세요 ----

# train_script("models/taylor_nn/ann.py", "configs/taylor_root_ann.yaml", "results/taylor_nn/ann",
#              train_npz, val_npz, test_npz)

# train_script("models/taylor_nn/lstm.py", "configs/taylor_root_lstm.yaml", "results/taylor_nn/lstm",
#              train_npz, val_npz, test_npz)

# train_script("models/taylor_nn/mlp.py", "configs/taylor_root_mlp.yaml", "results/taylor_nn/mlp",
#              train_npz, val_npz, test_npz)

# train_script("models/transformer/model.py", "configs/transformer_interval.yaml", "results/transformer_interval",
#              interval_train_npz, interval_val_npz, interval_test_npz, extra_env={"MODE":"train"})


## 7) 평가 실행 (baseline 포함)
- `evaluation/evaluate_k_sweep.py` + `configs/eval_k_sweep.yaml`
- 실패 func_id 분포/히스토그램/박스플롯 등은 YAML 토글에 의해 저장됩니다.

In [ ]:
import torch
from pathlib import Path

eval_script = R("evaluation/evaluate_k_sweep.py")
assert eval_script and eval_script.exists(), f"Missing eval script: {eval_script}"

# 결과 저장 폴더(상대경로)
OUTDIR = R("results/runs_k_sweep_viz")
ensure_dir(OUTDIR)

env = os.environ.copy()
env["EVAL_CFG"]  = str(R("configs/eval_k_sweep.yaml"))
env["OUTDIR"]    = str(OUTDIR)
env["DEVICE"]    = ("cuda" if torch.cuda.is_available() else "cpu")
env["PYTHONPATH"]= str(REPO)

run([sys.executable, str(eval_script)], env=env, cwd=str(REPO))

print("\n[OK] Evaluation finished.")
print("Outputs under:", OUTDIR.relative_to(REPO))


## 8) 결과 파일 빠른 확인 (생성된 표/그림/리포트)

In [ ]:
from glob import glob

outdir = R("results/runs_k_sweep_viz")
pngs = sorted(glob(str(outdir / "*.png")))
csvs = sorted(glob(str(outdir / "*.csv")))
jsons = sorted(glob(str(outdir / "*.json")))

print("[PNG] ", len(pngs))
for p in pngs[:12]:
    print(" ", Path(p).relative_to(REPO))
if len(pngs) > 12:
    print("  ...")

print("\n[CSV] ", len(csvs))
for p in csvs[:12]:
    print(" ", Path(p).relative_to(REPO))
if len(csvs) > 12:
    print("  ...")

print("\n[JSON] ", len(jsons))
for p in jsons[:12]:
    print(" ", Path(p).relative_to(REPO))
if len(jsons) > 12:
    print("  ...")


## 9) (선택) baseline/모델로 예측이 안 되는 함수(func_id) 분포 확인
- `eval_k_sweep.yaml`에서 `reports.report_fail_funcid: true`일 때 저장된 JSON/CSV를 읽습니다.

In [ ]:
import pandas as pd

# 저장된 fail_by_funcid_*.csv 중 하나를 자동으로 찾음
outdir = R("results/runs_k_sweep_viz")
cands = sorted(outdir.glob("fail_by_funcid_*.csv"))

if not cands:
    print("[INFO] fail_by_funcid_*.csv not found. (reports.report_fail_funcid를 YAML에서 켜면 생성됩니다)")
else:
    p = cands[0]
    print("[LOAD]", p.relative_to(REPO))
    df = pd.read_csv(p)
    display(df.head(20))
